# Makata: first QLoRA engineering smoke test

Exactly 20 optimizer updates on 16 provisional Aya training examples. This does not establish improved comprehension, native-speaker review, legal clearance, or completion of Makata-v0.1.

Use **free T4 / standard RAM**. Stop if Colab asks for payment or cannot allocate a T4. Run cells in order. The notebook embeds source only; upload the private input ZIP separately. No evaluation data is needed.


## 1. Check hardware and install the separate pinned environment
May take several minutes. Setup time is excluded from training measurements.


In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, uuid, zipfile

# Select Runtime > Change runtime type > T4 GPU. Use the free standard runtime.
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
assert len(gpu.strip().splitlines()) == 1 and 'T4' in gpu, 'One free T4 required. No CPU/paid fallback.'
print(gpu)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '3b8fbb5fb55f493f1319f241081ee95a87cd443a3ed3f2f8891e5a9a24299150'
ROOT = pathlib.Path('/content') / ('makata-qlora-' + '3b8fbb5fb55f')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    receipts = json.loads(archive.read('source-manifest.json'))['sha256']
    assert set(archive.namelist()) == set(receipts) | {'source-manifest.json'}
    for name, expected in receipts.items():
        target = (ROOT / name).resolve()
        assert target.is_relative_to(ROOT.resolve())
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == expected
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
os.environ.update({'HF_HUB_DISABLE_TELEMETRY':'1', 'WANDB_DISABLED':'true', 'TOKENIZERS_PARALLELISM':'false'})
os.environ['HF_HOME'] = str(ROOT / 'research/.cache/huggingface')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.11.28'], check=True)
ENV = ROOT / 'research/.cache/qlora-venv'
if not ENV.exists():
    subprocess.run(['uv', 'venv', '--python', '3.12.13', str(ENV)], check=True)
PYTHON = ENV / 'bin/python'
subprocess.run(['uv', 'pip', 'sync', '--python', str(PYTHON), '--torch-backend', 'cu126',
                str(ROOT / 'research/requirements-qlora-colab.txt')], check=True)
subprocess.run(['uv', 'pip', 'check', '--python', str(PYTHON)], check=True)
SESSION = ROOT / 'research/runs' / ('smoke-' + uuid.uuid4().hex[:12])
SESSION.mkdir(parents=True)
CLI = ROOT / 'research/qlora_smoke.py'
def run(*arguments):
    command = [str(PYTHON), '-X', 'utf8', str(CLI), *map(str, arguments)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
run('preflight', '--output', SESSION / 'preflight')
print('Isolated environment and T4 verified. Source snapshot:', '3b8fbb5fb55f493f1319f241081ee95a87cd443a3ed3f2f8891e5a9a24299150')


## 2. Upload and verify the private inputs
Choose the ZIP produced by `qlora_smoke.py prepare`. The source text remains unchanged.


In [ ]:
from google.colab import files
UPLOAD = SESSION / 'private-inputs.zip'
sidebar_upload = pathlib.Path('/content/private-inputs.zip')
if sidebar_upload.exists():
    # Optional fallback: Files > Upload to session storage (no Drive mount).
    UPLOAD.write_bytes(sidebar_upload.read_bytes())
else:
    uploaded = files.upload()  # Select only the prepared private-inputs.zip.
    assert len(uploaded) == 1, 'Upload exactly one private input bundle.'
    UPLOAD.write_bytes(next(iter(uploaded.values())))
    del uploaded
INPUTS = SESSION / 'inputs'
run('unpack', '--archive', UPLOAD, '--output', INPUTS)
run('tokenize', '--inputs', INPUTS, '--output', SESSION / 'tokenization')
print('Exactly 16 frozen training records verified; no validation or benchmark data loaded.')


## 3. Test masking, data guards, and the training-library integration
The tiny CPU model checks code paths; it is not a GPU quantization test.


In [ ]:
subprocess.run([str(PYTHON), '-X', 'utf8', '-m', 'unittest', 'discover',
                '-s', 'research/tests', '-p', 'test_qlora_smoke.py', '-v'], cwd=ROOT, check=True)
run('integration', '--output', SESSION / 'cpu-integration')


## 4. Train, save, and reload in a fresh process
Only a finite 20-update run with matching reloaded adapter tensors and probe logits receives `complete` status. Do not repeatedly rerun to cherry-pick losses.


In [ ]:
RUN = SESSION / 'gpu-run'
RESULT_ZIP = SESSION / 'qlora-smoke-results.zip'
try:
    # Each command starts a new interpreter. Training exits before adapter reload.
    run('train', '--inputs', INPUTS, '--output', RUN)
    run('reload', '--output', RUN)
    run('verify', '--output', RUN)
finally:
    # Failures keep their incomplete manifest. An interrupted cell can use the next cell.
    if (RUN / 'run-manifest.json').exists() and not RESULT_ZIP.exists():
        run('archive', '--output', RUN, '--archive', RESULT_ZIP)


## 5. Download local evidence
Keep results private. Back on your computer, import the ZIP with `qlora_smoke.py import --archive ... --output ...`. A partial run remains incomplete.


In [ ]:
from google.colab import files
if not RESULT_ZIP.exists() and (RUN / 'run-manifest.json').exists():
    run('archive', '--output', RUN, '--archive', RESULT_ZIP)
print(json.loads((RUN / 'run-manifest.json').read_text())['status'])
files.download(str(RESULT_ZIP))
# Download before disconnecting. No Drive mount, hub upload, or paid service is used.
